# 05 - 完整集成：从零到一的 vLLM Triton 算子开发流程

> **本 Notebook 涵盖内容**
> - 完整的端到端开发流程回顾
> - 从 Triton kernel 编写到模型级替换的完整代码
> - 构建一个可复用的 Triton 算子库
> - 性能 benchmark 与分析
> - 实际开发中的注意事项与最佳实践
> - 全教程知识图谱

**运行示例**：将前 4 章的所有知识整合为一个完整的、可运行的算子开发流程。

## 1. 全教程知识图谱

![全教程知识图谱](../diagrams/05-knowledge-map.svg)

## 2. 完整的算子开发流程

下面我们从头演示一个完整的、生产级的自定义算子开发流程：

**目标**：为 vLLM 添加一个带 clamping 的 GeGLU 激活函数的 Triton 实现。

$$\text{GeGLU}_{\text{clamped}}(x) = \text{GELU}(\text{clamp}(x_{[:d]}, -L, L)) \cdot \text{clamp}(x_{[d:]}, -L, L)$$

**为什么选择这个例子？** 因为它结合了：
1. 非线性激活（GELU）
2. 数值稳定性处理（clamp）
3. 门控机制（gate * up）
4. 实际使用场景（Gemma3n 等模型使用类似结构）

### Step 1: 编写 PyTorch 参考实现

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import triton
import triton.language as tl
import math
from typing import Optional

device = torch.device("cuda")

def pytorch_clamped_geglu(x: torch.Tensor, limit: float = 7.0) -> torch.Tensor:
    """
    PyTorch 参考实现 — 用于验证 Triton 版本的正确性

    这是我们的 ground truth。
    """
    d = x.shape[-1] // 2
    gate = x[..., :d].clamp(-limit, limit)
    up = x[..., d:].clamp(-limit, limit)
    return F.gelu(gate) * up


# 验证参考实现
x = torch.randn(4, 512, device=device, dtype=torch.float16)
result = pytorch_clamped_geglu(x)
print(f"Reference implementation: input {x.shape} -> output {result.shape}")
print(f"Output range: [{result.min().item():.3f}, {result.max().item():.3f}]")

### Step 2: 编写 Triton Kernel

GELU 的精确公式：
$$\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$$

Triton 没有内置 `erf`，我们使用 tanh 近似：
$$\text{GELU}(x) \approx 0.5 x \left[1 + \tanh\left(\sqrt{\frac{2}{\pi}}(x + 0.044715 x^3)\right)\right]$$

**数值示例**：当 x = 1.0 时：
- $\sqrt{2/\pi} \approx 0.7979$
- 内部项 = $0.7979 \times (1.0 + 0.044715) = 0.8336$
- $\tanh(0.8336) = 0.6837$
- $\text{GELU}(1.0) = 0.5 \times 1.0 \times (1 + 0.6837) = 0.8419$

**生活类比**：GELU 比 ReLU 更「柔和」。如果 ReLU 是一扇门——要么全开要么全关，那 GELU 就是一扇自动感应门——接近的时候缓缓打开，走到正中间时半开，走远了才全开。Clamp 就是给门加了限位器，防止它开得太大或关得太紧。

In [ ]:
SQRT_2_OVER_PI = math.sqrt(2.0 / math.pi)  # 0.7978845608

@triton.autotune(
    configs=[
        triton.Config({"BLOCK_SIZE": 256}, num_warps=2),
        triton.Config({"BLOCK_SIZE": 512}, num_warps=4),
        triton.Config({"BLOCK_SIZE": 1024}, num_warps=4),
        triton.Config({"BLOCK_SIZE": 2048}, num_warps=8),
    ],
    key=["d"],
)
@triton.jit
def clamped_geglu_kernel(
    output_ptr, o_stride,
    input_ptr, i_stride,
    d,
    limit: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    """
    融合的 Clamped GeGLU Triton kernel

    一次性完成: clamp + GELU + gate * up
    数据只从 HBM 读一次，写一次。
    """
    row = tl.program_id(axis=0)
    col_block = tl.program_id(axis=1)

    offsets = col_block * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < d

    inp = input_ptr + row * i_stride
    out = output_ptr + row * o_stride

    # 加载 gate 和 up
    gate = tl.load(inp + offsets, mask=mask).to(tl.float32)
    up = tl.load(inp + offsets + d, mask=mask).to(tl.float32)

    # Clamp
    gate = tl.minimum(tl.maximum(gate, -limit), limit)
    up = tl.minimum(tl.maximum(up, -limit), limit)

    # GELU (tanh approximation)
    # gelu(x) = 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
    c = 0.7978845608  # sqrt(2/pi)
    inner = c * (gate + 0.044715 * gate * gate * gate)
    # tanh via: tanh(x) = 2*sigmoid(2x) - 1
    gate_gelu = 0.5 * gate * (1.0 + tl.extra.libdevice.tanh(inner))

    result = gate_gelu * up
    tl.store(out + offsets, result.to(input_ptr.dtype.element_ty), mask=mask)


def triton_clamped_geglu(x: torch.Tensor, limit: float = 7.0) -> torch.Tensor:
    if x.ndim != 2:
        orig_shape = x.shape
        x = x.view(-1, x.shape[-1])
    else:
        orig_shape = None

    b, n = x.shape
    d = n // 2
    output = torch.empty((b, d), dtype=x.dtype, device=x.device)
    grid = lambda meta: (b, triton.cdiv(d, meta["BLOCK_SIZE"]))

    clamped_geglu_kernel[grid](
        output, output.stride(0),
        x, x.stride(0),
        d, limit=limit,
    )
    if orig_shape is not None:
        return output.view(orig_shape[:-1] + (d,))
    return output


# 验证 Triton vs PyTorch
x = torch.randn(128, 8192, device=device, dtype=torch.float16)
result_triton = triton_clamped_geglu(x)
result_pytorch = pytorch_clamped_geglu(x)

max_err = (result_triton - result_pytorch).abs().max().item()
mean_err = (result_triton - result_pytorch).abs().mean().item()
print(f"Clamped GeGLU Triton kernel verification:")
print(f"  Max error:  {max_err:.2e}")
print(f"  Mean error: {mean_err:.2e}")
assert max_err < 5e-2, f"Error too large: {max_err}"
print("  PASSED!")

### Step 3: 注册为 CustomOp

In [ ]:
# ==== 完整的注册系统 ====
op_registry = {}
op_registry_oot = {}

class CustomOp(nn.Module):
    def __new__(cls, *args, **kwargs):
        op_name = cls.__name__
        if op_name in op_registry_oot:
            return super().__new__(op_registry_oot[op_name])
        return super().__new__(cls)

    def __init__(self, **kw):
        super().__init__()
        self._forward_method = self.forward_cuda if torch.cuda.is_available() else self.forward_native

    def forward(self, *args, **kwargs):
        return self._forward_method(*args, **kwargs)

    def forward_native(self, *args, **kwargs):
        raise NotImplementedError
    def forward_cuda(self, *args, **kwargs):
        raise NotImplementedError

    @classmethod
    def register(cls, name):
        def decorator(op_cls):
            op_cls.name = name
            op_registry[name] = op_cls
            return op_cls
        return decorator

    @classmethod
    def register_oot(cls, _cls=None, name=None):
        def decorator(op_cls):
            reg_name = name or cls.__name__
            op_registry_oot[reg_name] = op_cls
            return op_cls
        if _cls is None:
            return decorator
        return decorator(_cls)


@CustomOp.register("clamped_geglu")
class ClampedGeGLU(CustomOp):
    """
    带 clamping 的 GeGLU 激活函数

    模式参照 vLLM 的 SwigluStepAndMul (activation.py:341-375)
    """

    def __init__(self, limit: float = 7.0, **kw):
        super().__init__(**kw)
        self.limit = limit

    def forward_native(self, x: torch.Tensor) -> torch.Tensor:
        return pytorch_clamped_geglu(x, self.limit)

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        return triton_clamped_geglu(x, self.limit)


# 注册一些辅助算子
@CustomOp.register("silu_and_mul")
class SiluAndMul(CustomOp):
    def __init__(self, **kw):
        super().__init__(**kw)
    def forward_native(self, x):
        d = x.shape[-1] // 2
        return F.silu(x[..., :d]) * x[..., d:]
    def forward_cuda(self, x):
        return self.forward_native(x)

@CustomOp.register("rms_norm")
class RMSNorm(CustomOp):
    def __init__(self, hidden_size, eps=1e-6, **kw):
        super().__init__(**kw)
        self.hidden_size = hidden_size
        self.variance_epsilon = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))
    def forward_native(self, x, residual=None):
        if residual is not None:
            x = x + residual
            residual = x
        x_f = x.float()
        v = x_f.pow(2).mean(-1, keepdim=True)
        x_out = (x_f * torch.rsqrt(v + self.variance_epsilon) * self.weight.float()).to(x.dtype)
        return (x_out, residual) if residual is not None else x_out
    def forward_cuda(self, x, residual=None):
        return self.forward_native(x, residual)


# 测试
op = ClampedGeGLU(limit=7.0)
x = torch.randn(128, 8192, device=device, dtype=torch.float16)
result = op(x)
print(f"ClampedGeGLU CustomOp:")
print(f"  Name: {op.name}")
print(f"  Dispatch: {op._forward_method.__name__}")
print(f"  Output shape: {result.shape}")
print(f"  Registered ops: {list(op_registry.keys())}")

### Step 4: 在模型中使用

In [ ]:
class LlamaMLP(nn.Module):
    """
    支持多种激活函数的 MLP 层

    对应 vllm/model_executor/models/llama.py:81-121
    """
    ACT_FN_MAP = {
        "silu": lambda: SiluAndMul(),
        "geglu": lambda: ClampedGeGLU(limit=7.0),
    }

    def __init__(self, hidden_size, intermediate_size, hidden_act="silu"):
        super().__init__()
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        if hidden_act not in self.ACT_FN_MAP:
            raise ValueError(f"Unsupported activation: {hidden_act}")
        self.act_fn = self.ACT_FN_MAP[hidden_act]()

    def forward(self, x):
        x = self.gate_up_proj(x)
        x = self.act_fn(x)
        x = self.down_proj(x)
        return x


class LlamaDecoderLayer(nn.Module):
    def __init__(self, hidden_size, intermediate_size, hidden_act="silu"):
        super().__init__()
        self.input_layernorm = RMSNorm(hidden_size)
        self.post_attention_layernorm = RMSNorm(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size, hidden_act)
        self.self_attn = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, hidden_states, residual=None):
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states, residual = self.input_layernorm(hidden_states, residual)
        hidden_states = self.self_attn(hidden_states)
        hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class MiniLlama(nn.Module):
    """完整的小型 Llama 模型"""
    def __init__(self, hidden_size=256, intermediate_size=512,
                 num_layers=4, vocab_size=1000, hidden_act="silu"):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(hidden_size, intermediate_size, hidden_act)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.embed(input_ids)
        residual = None
        for layer in self.layers:
            h, residual = layer(h, residual)
        h = self.norm(h + residual if residual is not None else h)
        return self.lm_head(h)


# 测试两种激活函数
for act in ["silu", "geglu"]:
    model = MiniLlama(hidden_act=act).half().to(device)
    input_ids = torch.randint(0, 1000, (2, 16), device=device)
    logits = model(input_ids)
    print(f"MiniLlama (act={act}): input {input_ids.shape} -> logits {logits.shape}")
    assert not torch.isnan(logits).any()
    loss = F.cross_entropy(logits.view(-1, 1000), input_ids.view(-1))
    loss.backward()
    print(f"  Loss: {loss.item():.4f}, backward: OK")

### Step 5: 性能 Benchmark

In [ ]:
# 全面性能对比
print("=" * 70)
print("Performance Benchmark: Triton Fused vs PyTorch Unfused")
print("=" * 70)

sizes = [
    (32, 2048),
    (64, 4096),
    (128, 8192),
    (128, 22016),  # Llama-7B intermediate_size
    (256, 22016),
]

print(f"\n{'Batch':>6} x {'2*d':>6} | {'PyTorch':>10} | {'Triton':>10} | {'Speedup':>8}")
print("-" * 55)

for b, two_d in sizes:
    x = torch.randn(b, two_d, device=device, dtype=torch.float16)

    t_pt = triton.testing.do_bench(lambda: pytorch_clamped_geglu(x))
    t_tr = triton.testing.do_bench(lambda: triton_clamped_geglu(x))
    speedup = t_pt / t_tr

    print(f"{b:>6} x {two_d:>6} | {t_pt:>8.4f}ms | {t_tr:>8.4f}ms | {speedup:>6.2f}x")

# 模型级 benchmark
print(f"\n{'='*70}")
print("Model-level Benchmark: MiniLlama (4 layers)")
print(f"{'='*70}")

model_silu = MiniLlama(hidden_size=512, intermediate_size=1024, hidden_act="silu").half().to(device)
model_geglu = MiniLlama(hidden_size=512, intermediate_size=1024, hidden_act="geglu").half().to(device)
input_ids = torch.randint(0, 1000, (8, 64), device=device)

t_silu = triton.testing.do_bench(lambda: model_silu(input_ids))
t_geglu = triton.testing.do_bench(lambda: model_geglu(input_ids))

print(f"SiluAndMul (native):      {t_silu:.4f} ms")
print(f"ClampedGeGLU (Triton):    {t_geglu:.4f} ms")

## 3. 开发最佳实践

### 3.1 正确性优先

```python
# 永远先写 forward_native，它是 ground truth
def forward_native(self, x):
    """这个实现必须 100% 正确，哪怕它很慢"""
    ...

# 然后写 forward_cuda，用 forward_native 验证
def forward_cuda(self, x):
    result = triton_kernel(x)
    # 开发时加断言
    if DEBUG:
        expected = self.forward_native(x)
        assert torch.allclose(result, expected, atol=1e-2)
    return result
```

### 3.2 数值精度

```python
# 在 Triton kernel 中：
x = tl.load(ptr + offsets, mask=mask).to(tl.float32)  # 升精度
result = complex_computation(x)                         # fp32 计算
tl.store(ptr + offsets, result.to(ptr.dtype.element_ty), mask=mask)  # 降回原精度
```

### 3.3 Grid 设计

```python
# 对于 (batch, hidden_size) 形状的操作：
grid = (batch_size, triton.cdiv(hidden_size, BLOCK_SIZE))

# 对于需要跨 hidden_size 规约的操作（如 RMSNorm）：
grid = (batch_size,)  # 只用 1D grid，在 kernel 内循环
```

### 3.4 性能调优

1. **选择正确的 BLOCK_SIZE**：使用 `@triton.autotune`
2. **最小化 HBM 访问**：在寄存器中完成尽可能多的计算
3. **避免 warp divergence**：确保同一 warp 内的线程走相同的分支
4. **利用内存合并**：连续线程访问连续内存地址

## 4. 常见陷阱与解决方案

| 陷阱 | 症状 | 解决方案 |
|------|------|---------|
| fp16 精度不足 | 大值输入时结果偏差大 | 在 kernel 内升到 fp32 计算 |
| BLOCK_SIZE 太大 | kernel launch 失败 | 使用 autotune 或限制 BLOCK_SIZE |
| 忘记 mask | 边界处产生垃圾值 | 所有 load/store 都加 mask |
| stride 计算错误 | 结果完全错误 | 使用 `tensor.stride(dim)` 而非手动计算 |
| OOT 注册顺序 | 替换没有生效 | 确保 OOT 注册在算子实例化之前 |
| Grid 维度错误 | 只处理了部分数据 | 使用 `triton.cdiv` 确保覆盖所有元素 |

## 5. 全教程总结

我们完成了从零到一的 vLLM Triton 算子开发之旅：

| 阶段 | 内容 | 关键技能 |
|------|------|---------|
| **Notebook 01** | Triton 基础 | `@triton.jit`, program_id, BLOCK_SIZE, mask |
| **Notebook 02** | CustomOp 注册 | `@CustomOp.register`, 双路径实现, 平台分发 |
| **Notebook 03** | 融合算子 | Memory-bound 分析, 算子融合, autotune |
| **Notebook 04** | 模型级替换 | OOT 机制, register_oot, 插件开发 |
| **Notebook 05** | 完整集成 | 端到端流程, benchmark, 最佳实践 |

### 三个核心场景

1. **添加自定义算子**：编写 Triton kernel + `@CustomOp.register` + forward_native/forward_cuda
2. **添加融合算子**：分析 memory-bound → 合并多个操作到单个 kernel → autotune
3. **替换模型算子**：`@CustomOp.register_oot` → 继承原始类 → 覆盖 forward_cuda

## 源码映射表（全教程）

| 本教程实现 | vLLM 原始源码 | 章节 |
|-----------|-------------|------|
| Triton kernel 基础 | `vllm/triton_utils/` | 01 |
| CustomOp 注册系统 | `vllm/model_executor/custom_op.py` | 02 |
| 融合 SiLU+Mul | `vllm/model_executor/layers/activation.py:26-74` | 03 |
| 融合 Add+RMSNorm | `vllm/model_executor/layers/layernorm.py:56-74` | 03 |
| OOT 替换机制 | `vllm/model_executor/custom_op.py:109-128, 332-353` | 04 |
| Llama MLP | `vllm/model_executor/models/llama.py:81-121` | 04, 05 |
| Llama DecoderLayer | `vllm/model_executor/models/llama.py:253-333` | 04, 05 |
| OOT 插件示例 | `tests/plugins/.../dummy_custom_ops.py` | 04 |
| Autotune | `vllm/model_executor/layers/mamba/ops/ssd_chunk_scan.py` | 03, 05 |
| ClampedGeGLU | `vllm/model_executor/layers/activation.py:341-375` | 05 |